# Small Hallbar Dunker Measurement

## Import Libraries

In [1]:
import time
import json
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from time import sleep

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.instrument.channel import ChannelList

from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq

from pyvisa.constants import StopBits, Parity

rm = pyvisa.ResourceManager()
rm.list_resources()

('ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR')

## Instantiation of Instruments

In [2]:
contacts = {
    "Gate" : 1,
    "D13" : 2,
    "D15" : 3,
    "D16" : 4,
    "D19" : 5,
    "D20" : 6,
    "D24" : 7,
}

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 0.5,
    contacts = contacts
)
station = Station(qdac2)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.09s


{'Gate': 0.0,
 'D13': 0.0,
 'D15': 0.0,
 'D16': 0.0,
 'D19': 0.0,
 'D20': 0.0,
 'D24': 0.0}

## QDAC2 Reset

In [3]:
qdac2.reset()

## Data Base Setup

In [3]:
# Set up database
initialise_or_create_database_at("./shb_data.db")

## Instrument Setup

In [160]:
## setup qdac 2 for the 2D sweep
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 15

acq_time = 200e-6
num_of_samples = int(acq_time * daq.max_sampling_rate)

qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")

exp = load_or_create_experiment("2D sweep", "shb_data_third_preamp_D10_D11_gate_float")
meas = Measurement(exp=exp, station=station)

ds_current = Parameter(name= "ds_current", label="Drain-Source Current", unit="nA")
bias = Parameter(name = "bias", label="Bias Voltage", unit = "uV" )
gate = Parameter(name = "gate",label="Gate Voltage", unit = "V" )

meas.register_parameter(bias)
meas.register_parameter(gate)
meas.register_parameter(ds_current, setpoints = (bias, gate))

# Measurement

In [161]:
initial_conditions= qdac2.get_initial_voltages()

gate_sweep = np.linspace(-1,1,21)
bias_sweep = np.linspace(-100e-6, 100e-6, 21)

start_time = time.time()

with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(
        tag = "Contacts",
        metadata = json.dumps(contacts)
    )
    datasaver.dataset.add_metadata(
        tag = "IC",
        metadata = json.dumps(initial_conditions)
    )
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "gate_chan": 1,
                "gate_start": gate_sweep[0],
                "gate_end": gate_sweep[-1],
                "bias_start": bias_sweep[0],
                "bias_end": bias_sweep[-1],
                "acq_time": acq_time,
            }
        )
    )
    for v_gate in gate_sweep:
        for v_bias in bias_sweep:
            qdac2.ramp_channels(["Gate"], [v_gate])
            qdac2.ramp_channels(["D13"], [v_bias])
            v_meas = np.array(
                daq.read(
                    ch = "Dev2/ai0",
                    num_of_samples = num_of_samples
                )
            ).mean(axis = -1)
            i_meas = daq.convert_volts_to_amps(v_meas)

            datasaver.add_result(
                (gate, v_gate),
                (bias, v_bias),
                (ds_current, i_meas)
            )
            loop_counter = loop_counter+1
        print(
            f"Time elapsed: {np.round(time.time()-start_time, 2)} sec."
            f" Loop finished: {loop_counter}/{len(gate_sweep) * len(bias_sweep)}."
        )
end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)
qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 72. 
Time elapsed: 1.8 sec. Loop finished: 21/441.
Time elapsed: 2.69 sec. Loop finished: 42/441.
Time elapsed: 3.56 sec. Loop finished: 63/441.
Time elapsed: 4.53 sec. Loop finished: 84/441.
Time elapsed: 5.37 sec. Loop finished: 105/441.
Time elapsed: 6.25 sec. Loop finished: 126/441.
Time elapsed: 7.1 sec. Loop finished: 147/441.
Time elapsed: 8.05 sec. Loop finished: 168/441.
Time elapsed: 8.9 sec. Loop finished: 189/441.
Time elapsed: 9.87 sec. Loop finished: 210/441.
Time elapsed: 10.71 sec. Loop finished: 231/441.
Time elapsed: 11.62 sec. Loop finished: 252/441.
Time elapsed: 12.49 sec. Loop finished: 273/441.
Time elapsed: 13.41 sec. Loop finished: 294/441.
Time elapsed: 14.3 sec. Loop finished: 315/441.
Time elapsed: 15.16 sec. Loop finished: 336/441.
Time elapsed: 16.08 sec. Loop finished: 357/441.
Time elapsed: 16.95 sec. Loop finished: 378/441.
Time elapsed: 17.87 sec. Loop finished: 399/441.
Time elapsed: 18.73 sec. Loop finished: 420/441